# Laboration 1: Intelligenta agenter


Begin by running the two example agents `RandomAgent` and `ReflexAgentWithState`. Try to understand what the different parts of the code does. Stop the agent using the `Stop` button above or by closing the Pacman window.

The actions that can be performed by a Pacman agent are: 
- *GoForward*: Take one step forward
- *GoRight*: Turn 90 degrees right and take one step forward
- *GoLeft*: Turn 90 degrees left and take one step forward
- *GoBack*: Turn and 180 degrees and take one step forward
- *Stop*: Shut down the agent
- Any other command: No effect

*Note: If the the Pacman window freezes, you can still (usually) re-run the code. If this does not work, try `Kernel/Restart`, or restart the Jupyter Notebook* 

In [2]:
from agents import *
from pacman import *

In [3]:
class RandomAgent(BaseAgent):

    class State:
        def __init__(self):
            self.actions = ["GoRight", "GoLeft", "GoForward", "GoBack"]

    def choose_action(self, state):
        action = random.choice(state.actions)
        print("Performing action:", action)
        return action

In [4]:
# Run RandomAgent in the room layout "layouts/custom.lay"
args = readCommand(["--pacman", RandomAgent,
                    "--layout", "mediumEmpty"])
runGames(**args)

Performing action: GoLeft
Performing action: GoLeft
Performing action: GoLeft
Performing action: GoRight
Performing action: GoLeft
Performing action: GoRight
Performing action: GoRight
Performing action: GoLeft
Performing action: GoLeft
Performing action: GoRight
Performing action: GoBack
Performing action: GoForward
Performing action: GoRight
Performing action: GoBack
Performing action: GoBack
Performing action: GoForward
Performing action: GoLeft
Performing action: GoLeft
Performing action: GoRight
Performing action: GoBack
Performing action: GoLeft
Performing action: GoRight
Performing action: GoRight
Performing action: GoBack
Performing action: GoLeft
Performing action: GoForward
Performing action: GoBack
Performing action: GoBack
Performing action: GoLeft
Performing action: GoLeft
Performing action: GoBack
Performing action: GoBack
Performing action: GoRight
Performing action: GoBack
Performing action: GoRight
Performing action: GoLeft
Performing action: GoRight
Performing action:

SystemExit: 0

/home/alfsj019/Desktop/729g78/lab2/venv/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3441: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
class ReflexAgentWithState(BaseAgent):
    class State:
        def __init__(self):
            self.bump = False
            self.previous_action = ""
            self.actions = ["GoRight", "GoLeft", "GoForward", "GoBack"]

        def __repr__(self):
            if self.bump:
                return self.previous_action + " resulted in a bump"
            else:
                return self.previous_action
    
    def update_state_with_percept(self, percept, state):
        if percept[1] == "bump":
            state.bump = True
        else:
            state.bump = False
        return state

    def choose_action(self, state):
        actions = state.actions
        if state.bump:
            actions.remove(state.previous_action)
        return random.choice(actions)

    def update_state_with_action(self, action, state):
        state.previous_action = action
        # Print the representation (i.e. __repr__) of the state
        print(state)
        return state

In [7]:
# Run ReflexAgentWithState
args = readCommand(["--pacman", ReflexAgentWithState,
                    "--layout", "mediumEmpty"])
runGames(**args)


    while executing
"4606708992_destroy_window"
    (command for "WM_DELETE_WINDOW" window manager protocol)
invalid command name "4602303680quit"
    while executing
"4602303680quit"
    ("after" script)


GoLeft
GoRight
GoRight
GoLeft
GoBack
GoLeft
GoRight resulted in a bump
GoLeft
GoBack resulted in a bump
GoBack
GoRight
GoLeft resulted in a bump
GoRight resulted in a bump
GoForward resulted in a bump
GoLeft resulted in a bump
GoBack resulted in a bump
GoRight
GoForward
GoBack
GoLeft
GoRight
GoForward
GoRight resulted in a bump
GoForward
GoLeft
GoRight resulted in a bump
GoRight
GoBack
GoLeft
GoLeft
GoRight
GoLeft
GoLeft
GoRight
GoRight
GoBack
GoBack
GoLeft
GoForward
GoRight
GoLeft
GoLeft
GoRight
GoForward
GoLeft
GoForward
GoRight
GoForward
GoLeft
GoBack
GoLeft
GoBack
GoForward
GoBack
GoRight
GoLeft
GoForward
GoLeft
GoRight
GoBack
GoBack
GoLeft
GoForward
GoLeft
GoForward
GoLeft
GoBack
GoForward
GoForward
GoBack resulted in a bump
GoLeft
GoForward
GoBack
GoRight
GoForward
GoRight resulted in a bump
GoBack
GoForward
GoForward
GoBack
GoForward
GoForward
GoBack
GoBack
GoRight
GoForward
GoLeft
GoLeft
GoRight
GoForward
GoBack
GoForward
GoRight
GoBack
GoLeft
GoForward
GoForward
GoBack
GoForwa

SystemExit: 0

**Exercise 1**: Briefly describe the behavior of the two agents above. What is the main difference between the two?

**Answer:** ...

**Exercise 2**: Devise a strategy that will allow the agent to eat all the food in the room and then stop. Discuss the strategy with your partner. Remember to test your strategy in 'odd' worlds, such as worlds of the size 1x1 or 1x5. Describe your strategy in general terms below.

**Answer:** ...

In [62]:
class AgentWithState(BaseAgent):

    class State:
        def __init__(self):
            
            self.bump = False
            self.previous_action = ""

            self.map_len = 0
            self.map_height = 0

            self.step_counter = 0
            self.step_max = 0

            self.line_counter = 0
            self.line_max = 0

            self.latest_turn = "Right"
            self.initial_left_turn = None
            self.is_turning = False

            self.phase = "length_scan"

        def __repr__(self):
            if self.bump:
                return self.previous_action + " resulted in a bump"
            else:
                return self.previous_action

    def update_state_with_percept(self, percept, state):
        """Update the state based on percept"""
        print(percept[1])
        if percept[1] and state.phase == "length_scan":
            state.bump = True
            state.phase = "height_scan"
            state.is_turning = True
            return state
        elif percept[1] and state.phase == "height_scan" and not state.is_turning:
            state.bump = True
            state.phase = "clear"
            state.is_turning = True
            return state 

        # elif percept[1] == "bump":
        #     state.bump = True
        #     return state

        elif state.map_height > state.map_len and state.phase == "clear":
            state.step_max = state.map_height -2
            state.line_max = state.map_len - 3
            state.latest_turn = "Left"
            state.initial_left_turn = True
        elif state.map_height < state.map_len and state.phase == "clear":
            state.step_max = state.map_len - 3
            state.line_max = state.map_height - 2
            state.latest_turn = "Right"
        else:
            state.bump = False
            state.is_turning = False
            return state
        

    def choose_action(self, state: State):
        """Return an action"""
        print(state.phase)

        if state.phase == "length_scan":
            self.update_state_with_percept(self._percept, self._state)
            if state.bump:
                self.update_state_with_action("GoLeft", self._state)
                # state.bump = False
                # state.phase = "height_scan"
                return "GoLeft"
            else:
                self.update_state_with_action("GoForward", self._state)
                return "GoForward"
        
        if state.phase == "height_scan":
            self.update_state_with_percept(self._percept, self._state)

            if state.bump and state.is_turning:
                self.update_state_with_action("GoLeft", self._state)
                return "GoLeft"
            
            #if state.bump:
                # if state.map_len == 2:
                #     return "Stop"
                
                # if state.map_len == 1 or state.map_height == 1:
                #     return "Stop"
                # #state.phase = "clear"
                

                # if state.map_height > state.map_len:
                #     state.step_max = state.map_height - 2
                #     #state.line_max = state.map_len - 3
                #     #state.latest_turn = "Left"
                #     #state.initial_left_turn = True
                
                # else:
                    #state.step_max = state.map_len - 3
                    #state.line_max = state.map_height - 2
                    #state.latest_turn = "Right"
            else:
                return "GoForward"
        
        if state.phase == "clear":

            if state.map_len == 1 or state.map_height == 1:
                return "Stop"
            
            if state.bump and state.is_turning:
                self.update_state_with_action("GoLeft", self._state)
                return "GoLeft"
        
            return "GoForward"



        
                

                # self.update_state_with_action("GoLeft", self._state)
                # #state.bump = False
                # return "GoLeft"

                #self.update_state_with_action("GoForward", self._state)
        #return "GoForward"

            # print("hej")
            # if state.step_max == 0:
                
            
            # if state.step_counter == state.step_max:
            #     if state.line_counter == state.line_max:
            #         return "Stop"
            #                     # state.bump = False
            #     # state.phase = "height_scan"tep_counter = 0

            #     state.is_turning = True
                    
            #     if state.latest_turn == "Right":
            #         self.update_state_with_action("GoLeft", self._state)
            #         return "GoLeft"
            #     elif state.latest_turn == "Left":
            #         self.update_state_with_action("GoRight", self._state)
            #         return "GoRight"
            #     # state.bump = False
            #     # state.phase = "height_scan"
            # if state.is_turning:
            #     state.is_turning = False

            #     if state.latest_turn == "Right":
            #         state.latest_turn = "Left"
            #         self.update_state_with_action("GoLeft", self._state)
            #         return "GoLeft"
            #     elif state.latest_turn == "Left":
            #         state.latest_turn = "Right"
            #         self.update_state_with_action("GoRight", self._state)
            #         return "GoRight"
            
            # if state.initial_left_turn:
            #     state.initial_left_turn = False
            #     self.update_state_with_action("GoLeft", self._state)
            #     return "GoLeft"
            
            # self.update_state_with_action("GoForward", self._state)
            # return "GoForward"
            

    def update_state_with_action(self, action, state):
        """Update the state based on the action performed"""
        #print("Performed action:", action)
        if state.phase == "length_scan":
            state.map_len += 1
        elif state.phase == "height_scan":
            state.map_len += 1
        
        # elif state.phase == "clear":
        #     state.line_counter += 1
        #     state.step_counter += 1
        state.previous_action = action
        return state

In [63]:
# Run AgentWithState. Try different layouts in the layout directory.
#Consider adding your own layouts to the ./layout directory to debug your code.
args = readCommand(["--pacman", AgentWithState,
                    "--layout", "mediumEmpty"])
runGames(**args)

None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
None
length_scan
None
bump
height_scan
bump
None
height_scan
None
None
height_scan
None
None
height_scan
None
None
height_scan
None
None
height_scan
None
None
height_scan
None
bump
clear
None


AttributeError: 'NoneType' object has no attribute 'phase'

**Exercise 3**: Implement the strategy you described in the previous exercise by extending `AgentWithState` above. Remember to comment and format the code appropriately.

**VG only:** Make sure your strategy prevents your agent from bumping more times than strictly necessary (beware of layouts 1x10, 10x1, 1x1).

**Exercise 4:** What environments are the agent best suited for? Describe the environment using terminology from the course literature (see Russel and Norvig, ch. 2).

**Answer:** ...

**Exercise 5**: Compare your agent to the predefined ones. How do they differ? Is your agent more intelligent than the other two? Is it more rational? Motivate your answers.

**Answer:** ...

**Exercise 6 (VG):** Discuss whether your agent's strategy is optimal or not. Could it be improved in any way given the current restrictions?

**Answer:** ...